# =========================================================
# BRAIN TUMOR CLASSIFICATION - FINAL UPDATED VERSION
#
# Dataset:
# https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri
#
# ZIP FILE PATH:
# /content/drive/MyDrive/Datasets/brain_tumor_detection.zip
#
# FEATURES:
# - Better Test Accuracy
# - Reduced Overfitting
# - Fine Tuning
# - MobileNetV2
# - GPU Optimized
# =========================================================

In [ ]:
# =========================================================
# 1. IMPORT LIBRARIES
# =========================================================

import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing import image

# =========================================================
# 2. CHECK TENSORFLOW VERSION
# =========================================================
print('TensorFlow Version:', tf.__version__)

In [ ]:
# =========================================================
# 3. MOUNT GOOGLE DRIVE
# =========================================================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =========================================================
# 4. CHECK GPU
# =========================================================
print('\nGPU STATUS:\n')
print(tf.config.list_physical_devices('GPU'))

# =========================================================
# 5. UNZIP DATASET
# =========================================================
zip_path     = '/content/drive/MyDrive/Datasets/brain_tumor_detection.zip'
extract_path = '/content/brain_tumor_dataset'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print('\nDataset Extracted Successfully')

# =========================================================
# 6. DATASET PATHS
# =========================================================
train_dir = os.path.join(extract_path, 'Training')
test_dir  = os.path.join(extract_path, 'Testing')

print('Train:', train_dir)
print('Test: ', test_dir)
print('Classes:', os.listdir(train_dir))

In [ ]:
# =========================================================
# 7. IMAGE SETTINGS
# =========================================================
IMG_SIZE   = 224
BATCH_SIZE = 32

# =========================================================
# 8. DATA AUGMENTATION
# =========================================================
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

# =========================================================
# 9. LOAD TRAIN DATA
# =========================================================
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

# =========================================================
# 10. LOAD VALIDATION DATA
# =========================================================
val_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=True
)

# =========================================================
# 11. LOAD TEST DATA
# =========================================================
test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =========================================================
# 12. SHOW CLASSES
# =========================================================
print('\nCLASSES:\n')
print(train_data.class_indices)

In [ ]:
# =========================================================
# 13. LOAD PRETRAINED MODEL
# =========================================================
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# =========================================================
# 14. FINE TUNING — freeze earlier layers
# =========================================================
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# =========================================================
# 15. BUILD MODEL
# =========================================================
inputs  = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = base_model(inputs, training=True)
x       = GlobalAveragePooling2D()(x)
x       = Dropout(0.5)(x)
x       = Dense(128, activation='relu')(x)
x       = Dropout(0.4)(x)
outputs = Dense(train_data.num_classes, activation='softmax')(x)

model = Model(inputs, outputs)

# =========================================================
# 16. COMPILE MODEL
# =========================================================
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# =========================================================
# 17. MODEL SUMMARY
# =========================================================
model.summary()

In [ ]:
# =========================================================
# 18. CALLBACKS
# =========================================================
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2
)

# =========================================================
# 19. TRAIN MODEL
# =========================================================
EPOCHS = 20

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
# =========================================================
# 20. EVALUATE MODEL
# =========================================================
test_loss, test_acc = model.evaluate(test_data)

print('\n=====================================')
print('TEST ACCURACY:', test_acc * 100)
print('=====================================')

# =========================================================
# 21. PLOT RESULTS
# =========================================================
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Train', 'Validation'])

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Train', 'Validation'])

plt.show()

In [ ]:
# =========================================================
# 22. SAVE MODEL TO GOOGLE DRIVE
# =========================================================
import json

save_dir = '/content/drive/MyDrive/ml_models'
os.makedirs(save_dir, exist_ok=True)

model.save(f'{save_dir}/brain_model.h5')

with open(f'{save_dir}/brain_classes.json', 'w') as f:
    json.dump(train_data.class_indices, f)

print('\nMODEL SAVED SUCCESSFULLY')
print('brain_model.h5 ->', save_dir)
print('brain_classes.json ->', save_dir)
print('Classes:', train_data.class_indices)

In [ ]:
# =========================================================
# 23. PREDICT SINGLE IMAGE
# =========================================================
# CHANGE IMAGE PATH
img_path = '/content/sample.jpg'

img       = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

prediction      = model.predict(img_array)
predicted_index = np.argmax(prediction)
class_names     = list(train_data.class_indices.keys())

print('\n=====================================')
print('PREDICTED CLASS:')
print(class_names[predicted_index])
print('\nCONFIDENCE:')
print(np.max(prediction) * 100)
print('=====================================')